# BAIR world model — 1. Data preparation and exploration

**First draft based on the supplied project proposal.** Research question: does training on a model's own predictions reduce error accumulation compared with single-step training?

This notebook imports real BAIR Robot Pushing videos, creates clip-level train/validation/test splits, checks temporal alignment and data quality, and explores the copy-last-frame baseline. No synthetic data or experimental results are supplied.

Run notebooks in numerical order. The pipeline uses 64×64 RGB frames, the current frame plus two preceding frames, and four-dimensional actions. Predictions are conditional on a known future action sequence; this is video prediction, not autonomous action planning.

## Data source and setup

The [TensorFlow Datasets catalog](https://www.tensorflow.org/datasets/catalog/bair_robot_pushing_small) documents 43,264 training clips and 256 test clips. Its [official builder](https://github.com/tensorflow/datasets/blob/master/tensorflow_datasets/datasets/bair_robot_pushing_small/bair_robot_pushing_small_dataset_builder.py) documents 30-frame clips and the raw record format used below. Download and storage requirements are substantial; this notebook never downloads automatically.

Choose an input route in the configuration cell (Parquet is now the default):

* `parquet`: the smaller local subset created by notebook 00. This route does not use TensorFlow.

* `raw`: locally extracted original BAIR TFRecords, with `train/` and `test/` under `RAW_ROOT` (usually the extracted `softmotion30_44k` directory).
* `tfds`: an already prepared local TensorFlow Datasets copy. Set `ALLOW_DOWNLOAD=True` only if you deliberately want TFDS to download and prepare it.

Install the packages in `requirements.txt` in your notebook environment. TensorFlow is needed only for the raw/TFDS routes here; PyTorch is used for training. The selected camera is fixed across all experiments.

## Smaller download through Parquet (default)

Run `00_huggingface_to_parquet.ipynb` first. It streams selected [guyuchao/BAIR shards](https://huggingface.co/datasets/guyuchao/BAIR/tree/main) into a local Parquet subset containing auxiliary-camera MP4 bytes and the corresponding robot actions. `SOURCE="parquet"` reads this subset with no TensorFlow dependency and no automatic full BAIR download.

This notebook then decodes the Parquet videos into the existing per-clip NumPy training cache; both representations use disk space. MP4 compression may change pixel values relative to the original BAIR release. Report the source revision and subset counts, and keep these experiments separate from raw-pixel BAIR comparisons. The separate `bair_hf_aux1_v1` output directories prevent accidental reuse of earlier raw-data checkpoints. Use different directory names if switching sources.

The community file named `bair_val.json` lists paths under the original **test** directory. We preserve that test split and create validation clips from the training subset.

In [1]:
from pathlib import Path
import json, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from the repository root or its notebooks/ directory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed" / "bair_hf_aux1_v1"
ARTIFACTS = ROOT / "artifacts" / "bair_hf_aux1_v1"
SEED = 4620
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
SOURCE = "parquet"                     # "parquet", "raw", or "tfds"
RAW_ROOT = ROOT / "data" / "raw" / "softmotion30_44k"  # EDIT this path
PARQUET_ROOT = ROOT / "data" / "parquet" / "bair_hf_subset"
TFDS_ROOT = ROOT / "data" / "tensorflow_datasets"
ALLOW_DOWNLOAD = True
CAMERA = "image_aux1"
VALIDATION_FRACTION = 0.10
MAX_TRAIN_CLIPS = None                  # e.g. 1000 for an initial BAIR run
MAX_TEST_CLIPS = None                   # None keeps the full official test split
CONTEXT = 3
MAX_HORIZON = 10
assert CAMERA == "image_aux1", "Change the processed/artifact directory names for a different camera."
assert SOURCE in {"parquet", "raw", "tfds"}
assert 0 < VALIDATION_FRACTION < 1

## Read BAIR clips

Original records store raw RGB bytes rather than JPEG files. The reader follows the official TFDS builder's context-feature layout. `action[t]` is treated as the command associated with the transition from frame `t` to `t+1`; confirm this convention against the data release used and document any remapping before interpreting action effects. The final action has no next-frame target and is discarded.

Every clip stays within one split. Windows are constructed only after splitting, so overlapping windows cannot leak across train and validation.

In [ ]:
# TensorFlow is optional for the Parquet route.
if SOURCE in {"raw", "tfds"}:
    import tensorflow as tf
    tf.config.set_visible_devices([], "GPU")

def raw_clips(split, limit=None):
    folder = RAW_ROOT / split
    files = sorted(p for p in folder.rglob("*") if p.is_file() and "tfrecord" in p.name)
    if not files:
        raise FileNotFoundError(
            f"No BAIR TFRecords under {folder}. Set RAW_ROOT to the extracted "
            "softmotion30_44k directory, or use SOURCE='tfds'.")
    count = 0
    for path in files:
        for record_index, payload in enumerate(tf.data.TFRecordDataset([str(path)])):
            proto = tf.train.SequenceExample.FromString(bytes(payload.numpy()))
            features = proto.context.feature
            frames, actions = [], []
            for t in range(30):
                image_key, action_key = f"{t}/{CAMERA}/encoded", f"{t}/action"
                if image_key not in features or action_key not in features:
                    raise ValueError(f"Unexpected BAIR record layout in {path}: missing frame {t}")
                image = np.frombuffer(features[image_key].bytes_list.value[0], dtype=np.uint8)
                frames.append(image.reshape(64, 64, 3))
                actions.append(np.asarray(features[action_key].float_list.value, dtype=np.float32))
            source_id = f"raw/{split}/{path.relative_to(folder).as_posix()}#{record_index}"
            yield source_id, np.stack(frames), np.stack(actions)
            count += 1
            if limit is not None and count >= limit:
                return

def tfds_clips(split, limit=None):
    import tensorflow_datasets as tfds
    builder = tfds.builder("bair_robot_pushing_small", data_dir=str(TFDS_ROOT))
    if ALLOW_DOWNLOAD:
        builder.download_and_prepare()
    try:
        ds = builder.as_dataset(split=split, shuffle_files=False)
    except (AssertionError, FileNotFoundError) as exc:
        raise FileNotFoundError(
            "BAIR TFDS data is not prepared. Set TFDS_ROOT to your prepared copy, "
            "use raw TFRecords, or explicitly enable ALLOW_DOWNLOAD.") from exc
    if limit is not None:
        ds = ds.take(limit)
    for i, sample in enumerate(tfds.as_numpy(ds)):
        yield f"tfds/{split}/{i:06d}", sample[CAMERA], sample["action"]

def parquet_clips(split, limit=None):
    import io
    import av
    import pyarrow.parquet as pq
    paths = sorted(PARQUET_ROOT.glob("*.parquet"))
    if not paths:
        raise FileNotFoundError("Run notebook 00 to create the BAIR Parquet subset first.")
    count = 0
    for path in paths:
        parquet = pq.ParquetFile(path)
        for batch in parquet.iter_batches(batch_size=32):
            for row in batch.to_pylist():
                if row["source_split"] != split:
                    continue
                with av.open(io.BytesIO(row["video_aux1_mp4"]), format="mp4") as video:
                    frames = np.stack([frame.to_ndarray(format="rgb24") for frame in video.decode(video=0)])
                actions = np.asarray(row["actions"], dtype=np.float32)
                yield f"hf/{row['clip_id']}", frames, actions
                count += 1
                if limit is not None and count >= limit:
                    return

reader = {"raw": raw_clips, "tfds": tfds_clips, "parquet": parquet_clips}[SOURCE]

## Export clips and a reproducible manifest

Store one compressed NumPy archive per clip to avoid loading the entire dataset into memory. Existing files are checked for matching content. Re-running this cell does not silently replace a different dataset; use a new processed directory if the camera or source changes. Optional clip caps select a reproducible source-order subset and must be reported as a limitation.

In [ ]:
DATA.mkdir(parents=True, exist_ok=True)
rows, hashes = [], set()
for source_split, limit in [("train", MAX_TRAIN_CLIPS), ("test", MAX_TEST_CLIPS)]:
    if limit is not None and limit < 2:
        raise ValueError("Use at least two clips per source split.")
    for source_id, frames, actions in reader(source_split, limit):
        frames = np.asarray(frames)
        actions = np.asarray(actions, dtype=np.float32)
        assert frames.dtype == np.uint8 and frames.shape == (30, 64, 64, 3)
        assert actions.shape == (30, 4) and np.isfinite(actions).all()
        actions = actions[:-1].copy()  # a[t]: I[t] -> I[t+1]
        content_hash = hashlib.sha256(frames.tobytes() + actions.tobytes()).hexdigest()
        if content_hash in hashes:
            raise ValueError(f"Duplicate clip detected: {source_id}. Resolve before splitting.")
        hashes.add(content_hash)
        clip_id = hashlib.sha256(source_id.encode()).hexdigest()[:20]
        relative = f"clips/{clip_id}.npz"
        target = DATA / relative
        target.parent.mkdir(exist_ok=True)
        if target.exists():
            with np.load(target, allow_pickle=False) as old:
                assert np.array_equal(old["frames"], frames) and np.array_equal(old["actions"], actions),                     f"Existing clip differs: {target}. Use a new DATA directory."
        else:
            np.savez_compressed(target, frames=frames, actions=actions)
        rows.append(dict(clip_id=clip_id, source_id=source_id, source_split=source_split,
                         path=relative, frames=len(frames), content_sha256=content_hash))
manifest = pd.DataFrame(rows)
assert len(manifest) > 0 and manifest.clip_id.is_unique
train_ids = manifest.index[manifest.source_split.eq("train")].to_numpy()
assert len(train_ids) >= 2, "Need at least two training clips for a held-out validation split."
rng = np.random.default_rng(SEED)
shuffled = rng.permutation(train_ids)
n_val = min(len(train_ids) - 1, max(1, round(len(train_ids) * VALIDATION_FRACTION)))
manifest["split"] = manifest.source_split
manifest.loc[shuffled[:n_val], "split"] = "val"
assert set(manifest.split) == {"train", "val", "test"}
assert manifest.frames.min() >= CONTEXT + MAX_HORIZON
manifest = manifest.sort_values(["split", "clip_id"]).reset_index(drop=True)
manifest_path = DATA / "manifest.csv"
csv_text = manifest.to_csv(index=False)
if manifest_path.exists() and manifest_path.read_text() != csv_text:
    raise RuntimeError("A different manifest exists. Use a new DATA directory to preserve prior experiments.")
manifest_path.write_text(csv_text)
metadata = dict(source=SOURCE, camera=CAMERA, seed=SEED,
                validation_fraction=VALIDATION_FRACTION, max_train_clips=MAX_TRAIN_CLIPS,
                max_test_clips=MAX_TEST_CLIPS, context=CONTEXT, max_horizon=MAX_HORIZON,
                action_convention="action[t] maps frame[t] to frame[t+1]",
                manifest_sha256=hashlib.sha256(csv_text.encode()).hexdigest())
if SOURCE == "parquet" and (PARQUET_ROOT / "source.json").exists():
    metadata["parquet_provenance"] = json.loads((PARQUET_ROOT / "source.json").read_text())
(DATA / "metadata.json").write_text(json.dumps(metadata, indent=2))
display(manifest.groupby(["source_split", "split"]).size().rename("clips").to_frame())

## Training-only action statistics

Images are divided by 255 at loading time. Action standardization uses only training clips; neither validation nor test contributes to fitted statistics. Near-constant action dimensions receive a minimum scale to avoid division by zero.

In [ ]:
total = np.zeros(4, dtype=np.float64)
total_sq = np.zeros(4, dtype=np.float64)
n = 0
for row in manifest[manifest.split.eq("train")].itertuples():
    with np.load(DATA / row.path, allow_pickle=False) as clip:
        a = clip["actions"].astype(np.float64)
    total += a.sum(axis=0)
    total_sq += (a * a).sum(axis=0)
    n += len(a)
mean = total / n
std = np.sqrt(np.maximum(total_sq / n - mean**2, 0)).clip(min=1e-6)
np.savez(DATA / "action_stats.npz", mean=mean.astype(np.float32), std=std.astype(np.float32))
display(pd.DataFrame({"action_dimension": range(4), "train_mean": mean, "train_std": std}))

In [ ]:
# Explore training clips only; reserve the test set for the final evaluation.
example = manifest[manifest.split.eq("train")].iloc[0]
with np.load(DATA / example.path, allow_pickle=False) as clip:
    frames, actions = clip["frames"].copy(), clip["actions"].copy()
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, t in zip(axes.flat, range(0, 30, 3)):
    ax.imshow(frames[t]); ax.set_title(f"Frame {t}"); ax.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(9, 3))
plt.plot(actions)
plt.xlabel("Transition index t"); plt.ylabel("Raw action")
plt.legend([f"a[{i}]" for i in range(4)]); plt.show()

# A descriptive baseline on this training clip, not a held-out result.
anchor = CONTEXT - 1
baseline_mse = [np.mean((frames[anchor].astype(np.float32) / 255 -
                         frames[anchor+h].astype(np.float32) / 255)**2)
                for h in range(1, MAX_HORIZON + 1)]
plt.plot(range(1, MAX_HORIZON + 1), baseline_mse, marker="o")
plt.xlabel("Prediction horizon"); plt.ylabel("Copy-last-frame MSE")
plt.title("One training clip: descriptive baseline"); plt.show()

## Data review before training

Record the source release, actual clip counts, camera choice, any clip caps, and whether the action convention has been verified. Inspect several training clips for camera changes, corrupt frames, and action/frame offsets. A random clip split measures performance in the same collection setting; it does not establish generalization to another robot or environment.

**Next:** run `02_bair_model_training.ipynb`. All reported model performance remains pending until real BAIR training and evaluation are completed.